# 04 · Backtest analysis

Load the two YUCLAW panels and explore hit rates / median returns. **Important caveat**: the in-sample panel was *reconstructed* via point-in-time replay, with market components running at 0.3 confidence. Treat it as a check on the **evidence layer** (C6 / C8 / C9), not as a validated live-trading backtest.

> **Disclaimer.** Research and education only. Not investment advice. Signal labels are research classifications, not buy/sell recommendations. YUCLAW is not a registered investment adviser. Past results — backtested or forward-tracked — do not predict future performance.

## Load both panels

In [ ]:
import yuclaw_py
import pandas as pd
import matplotlib.pyplot as plt

client = yuclaw_py.Client()
panels = client.backtest()
backtest, forward = panels['backtest'], panels['forward']
print(f"In-sample: {len(backtest)} rows ({backtest['signal_date'].min()} → {backtest['signal_date'].max()})")
print(f"Forward:   {len(forward)} rows ({forward['signal_date'].min()} → {forward['signal_date'].max() if len(forward) else 'n/a'})")

## Hit rate by label and horizon (in-sample)

In [ ]:
directional = ['STRONG_BUY', 'BUY', 'WEAKENING', 'NEGATIVE_EVENT', 'DOWNSIDE_WATCH']
sub = backtest[backtest['signal_label'].isin(directional)].copy()
summary = sub.groupby('signal_label').agg(
    n=('snapshot_id', 'count'),
    hit_1d=('hit_1d', 'mean'),
    hit_5d=('hit_5d', 'mean'),
    hit_20d=('hit_20d', 'mean'),
    median_5d=('return_5d', 'median'),
).round(3)
summary

## Median 5-day return by label

In [ ]:
ax = (sub.groupby('signal_label')['return_5d'].median() * 100).sort_values().plot(
    kind='barh', figsize=(8, 4), title='Median 5d return by label — in-sample (May 2026)')
ax.set_xlabel('median return %')
plt.tight_layout()

## Forward panel

Day 0 = 2026-05-20. Outcomes mature daily; expect this panel to be sparse for the first few weeks after launch.

In [ ]:
print(f'Forward snapshots: {len(forward)}')
print(f'Matured 1d:  {forward["return_1d"].notna().sum()}')
print(f'Matured 5d:  {forward["return_5d"].notna().sum()}')
print(f'Matured 20d: {forward["return_20d"].notna().sum()}')

---
> **Disclaimer.** Research and education only. Not investment advice. Signal labels are research classifications, not buy/sell recommendations. YUCLAW is not a registered investment adviser. Past results — backtested or forward-tracked — do not predict future performance.